# 第 4 章 残差の診断: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/04-residual-diagnostics.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0                # 乱数の種
n = 100                 # 標本の大きさ
heteroskedastic = True  # True: x が大きいほど誤差が大きい（扇形）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and Path(getattr(_lib, "__file__", "") or "").resolve() != (_root / "lib" / "__init__.py").resolve():
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## てこ比・スチューデント化残差・クックの距離

In [ ]:
import numpy as np

from lib.chapter03.multiple import fit_ols, leverage
from lib.chapter04.diagnostics import cooks_distance, studentized_residuals
from lib.chapter08.hetero import draw_fan_sample

x, y = draw_fan_sample(n, np.random.default_rng(seed), heteroskedastic)
result = fit_ols(y, x)
h = leverage(x)
print(f"傾き {result.coefficients.iloc[1]:.3f}、R² {result.r_squared:.3f}")
print(f"てこ比の最大 {h.max():.3f}")
print(f"スチューデント化残差の絶対値の最大 "
      f"{np.abs(studentized_residuals(result, h)).max():.3f}")
print(f"クックの距離の最大 {cooks_distance(result, h).max():.3f}")

## 図: 回帰の診断図

In [ ]:
from lib.chapter04.plot import plot_diagnostics

plot_diagnostics(y, x)

## 図: 要約統計量が同じで、形が違う 4 組のデータ

In [ ]:
from lib.chapter04.diagnostics import same_summary_datasets
from lib.chapter04.plot import plot_same_summary

plot_same_summary(same_summary_datasets())

## 試してみよう

- `heteroskedastic` を `False` にすると、誤差が均一な標本になり、R² は 0.596 から
  0.858、クックの距離の最大は 0.189 から 0.087 になります。診断図の残差の扇形も、
  消えます
- `n` を 400 にすると、てこ比の最大は 0.042 から 0.011
  に下がります（観測値が増えると、1 点が回帰に及ぼす力は、小さくなります）
- 4 組のデータの図は、数字を変えられません。平均・標準偏差・相関・回帰直線が同じでも、
  形が違うことを、確かめてください